# 03b · T2 ACT-Rの成分診断

主ACT-RとAudioCosineの結果を確認した後の追加診断です。主9種とLの決定を変更しません。Sのみ・Pのみ・organic再生だけから作るS+P、さらにraw成分と候補数を比較します。

部分一致はYambda `normalized_embed` の128次元音響CNNベクトル。AudioCosineと同じ埋め込みを使い、直前曲の文脈と全organic履歴平均、softmaxの有無を区別します。organic版では経路フィルタ後に30分セッションを再構成し、直前organic曲とorganic候補履歴を使います。全版の入力はstrictに起点前だけ、表現期限は29日/149日です。

In [1]:
import polars as pl
from IPython.display import display
from relisten import config
from relisten.actr_components import run_component_analysis
metrics,diagnostics = run_component_analysis()

In [2]:
display(metrics.filter((pl.col("route")=="C") & (pl.col("split")=="test") & (pl.col("lower_days")==config.LOWER_BOUND_DAYS) & (pl.col("protocol")=="score_only")))
display(diagnostics)
display(pl.read_csv(config.RESULTS / "actr_component_paired_vs_audio.csv"))

model,protocol,task,route,split,lower_days,is_main_lower,pairs,users,auc_users,positive_rate,user_auc,user_auc_ci_low,user_auc_ci_high,global_auc,pr_auc,log_loss,brier
str,str,str,str,str,i64,bool,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""ACT-R""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.51052,0.506406,0.515038,0.565916,0.125519,0.316887,0.087018
"""AudioCosine""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.561862,0.557,0.566788,0.596227,0.129059,0.31339,0.08637
"""ACT-R-S-only""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.503799,0.501355,0.506371,0.511405,0.101245,0.317006,0.087036
"""ACT-R-P-only""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.509981,0.505385,0.514604,0.560863,0.125153,0.316742,0.087
"""ACT-R-organic-SP""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.469392,0.464815,0.474187,0.54549,0.109158,0.315069,0.086772
"""ACT-R-S-raw""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.520177,0.516527,0.524005,0.502451,0.095826,0.316964,0.087031
"""ACT-R-P-raw""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.502654,0.498126,0.506986,0.497911,0.095461,0.317053,0.087047
"""ACT-R-candidate-count""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.529804,0.524468,0.535115,0.564026,0.126031,0.318395,0.087111


split,pairs,s_target_exact_zero_fraction,s_self_mass_above_099_fraction,s_max_mass_above_099_fraction,organic_context_fraction,partial_audio_available_fraction,organic_context_available_fraction,median_candidate_count
str,i64,f64,f64,f64,f64,f64,f64,f64
"""test""",967884,0.860192,0.742575,0.742807,0.035287,0.940268,0.93943,1579.0
"""train""",755797,0.925196,0.797508,0.797698,0.032027,0.931812,0.918657,752.0
"""validation""",283134,0.942525,0.803439,0.803701,0.033751,0.93584,0.937196,1430.0


model,reference,protocol,lower_days,task,route,split,paired_users,mean_auc_difference,difference_ci_low,difference_ci_high
str,str,str,i64,str,str,str,i64,f64,f64,f64
"""ACT-R""","""AudioCosine""","""score_only""",1,"""T2""","""C""","""test""",4124,-0.051342,-0.057052,-0.045272
"""ACT-R-S-only""","""AudioCosine""","""score_only""",1,"""T2""","""C""","""test""",4124,-0.058063,-0.063562,-0.05243
"""ACT-R-P-only""","""AudioCosine""","""score_only""",1,"""T2""","""C""","""test""",4124,-0.051881,-0.057661,-0.046258
"""ACT-R-organic-SP""","""AudioCosine""","""score_only""",1,"""T2""","""C""","""test""",4124,-0.09247,-0.100297,-0.084763
"""ACT-R-S-raw""","""AudioCosine""","""score_only""",1,"""T2""","""C""","""test""",4124,-0.041685,-0.047704,-0.035195
…,…,…,…,…,…,…,…,…,…,…
"""ACT-R-P-only""","""AudioCosine""","""baseline_plus_score""",14,"""T2""","""C""","""test""",3820,-0.052473,-0.058427,-0.046308
"""ACT-R-organic-SP""","""AudioCosine""","""baseline_plus_score""",14,"""T2""","""C""","""test""",3820,-0.092639,-0.10065,-0.084686
"""ACT-R-S-raw""","""AudioCosine""","""baseline_plus_score""",14,"""T2""","""C""","""test""",3820,-0.046317,-0.052331,-0.039956


診断用の予測・ユーザー別AUCは `data/processed/actr_diagnostics/` に隔離します。主比較の予測を上書きしません。L=7・14を含む全表は `results/actr_component_metrics.csv`、入力と埋め込みの定義は `results/actr_component_protocol.json` に保存します。

## ヘッドの向きと成分間の差

organic版はraw全体AUCとユーザー内AUCで向きが異なります。係数は学習だけで推定し、テストで符号を選び直しません。実測の解釈は `results/ACTR_COMPONENT_REPORT.md`。

In [3]:
display(pl.read_csv(config.RESULTS / "actr_component_head_directions.csv"))
display(pl.read_csv(config.RESULTS / "actr_organic_context_groups.csv"))
display(pl.read_csv(config.RESULTS / "actr_component_mechanism_differences.csv"))

model,split,lower_days,scaled_head_coefficient,head_direction,raw_global_auc,raw_user_auc
str,str,i64,f64,str,f64,f64
"""ACT-R""","""train""",1,0.033146,"""positive""",0.538625,0.504866
"""ACT-R""","""validation""",1,0.033146,"""positive""",0.555983,0.499974
"""ACT-R""","""test""",1,0.033146,"""positive""",0.565916,0.51052
"""AudioCosine""","""train""",1,0.400352,"""positive""",0.601474,0.561508
"""AudioCosine""","""validation""",1,0.400352,"""positive""",0.601729,0.568328
…,…,…,…,…,…,…
"""ACT-R-P-raw""","""validation""",1,-0.000572,"""negative""",0.505024,0.50074
"""ACT-R-P-raw""","""test""",1,-0.000572,"""negative""",0.502089,0.497346
"""ACT-R-candidate-count""","""train""",1,-0.135279,"""negative""",0.45856,0.4844


split,organic_context_available,pairs,positive_rate,median_organic_sp
str,bool,i64,f64,f64
"""train""",true,694318,0.095707,0.005606
"""test""",true,909259,0.100361,0.002443
"""test""",false,58625,0.032972,2.0
"""train""",false,61479,0.027717,2.0
"""validation""",true,265352,0.110107,0.002433
"""validation""",false,17782,0.03318,2.0


left_model,right_model,lower_days,task,route,split,protocol,paired_users,mean_auc_difference,difference_ci_low,difference_ci_high
str,str,i64,str,str,str,str,i64,f64,f64,f64
"""ACT-R""","""ACT-R-P-only""",1,"""T2""","""C""","""test""","""score_only""",4124,0.000539,-0.002124,0.003345
"""ACT-R-S-raw""","""ACT-R-S-only""",1,"""T2""","""C""","""test""","""score_only""",4124,0.016378,0.012383,0.020264
"""ACT-R-P-only""","""ACT-R-P-raw""",1,"""T2""","""C""","""test""","""score_only""",4124,0.007327,-0.000929,0.016253
"""ACT-R-candidate-count""","""ACT-R""",1,"""T2""","""C""","""test""","""score_only""",4124,0.019284,0.013218,0.0248
"""ACT-R""","""ACT-R-P-only""",3,"""T2""","""C""","""test""","""score_only""",4073,0.000025,-0.0029,0.002907
…,…,…,…,…,…,…,…,…,…,…
"""ACT-R-candidate-count""","""ACT-R""",7,"""T2""","""C""","""test""","""score_only""",3997,0.019063,0.013126,0.025487
"""ACT-R""","""ACT-R-P-only""",14,"""T2""","""C""","""test""","""score_only""",3820,-0.001183,-0.004394,0.001815
"""ACT-R-S-raw""","""ACT-R-S-only""",14,"""T2""","""C""","""test""","""score_only""",3820,0.015281,0.010962,0.019489
